# 🧠 06 · SHAP & Interpretability

> **Chapter 6.** What did the model actually learn? Which features
> drive each prediction? And where does it get confidently wrong?

TreeSHAP was run on a 2,000-variant stratified test sample (1,000
pathogenic + 1,000 benign, so the mean-|SHAP| ordering isn't biased
by the majority class).

---

In [ ]:
from __future__ import annotations
import sys
from pathlib import Path as _Path
sys.path.insert(0, str(_Path.cwd().parent))
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})

REPO = Path.cwd().parent
shap_df = pd.read_parquet(REPO / "results/metrics/shap_values_test.parquet")
feature_cols = [c for c in shap_df.columns if c != "variant_key"]
print(f"SHAP values shape: {shap_df.shape}")
print(f"Encoded features:  {len(feature_cols)}")

## Top-20 features by mean |SHAP|

In [ ]:
abs_shap = shap_df[feature_cols].abs()
mean_abs = abs_shap.mean(axis=0).sort_values(ascending=False)
top20 = mean_abs.head(20)
print(top20.to_string())

In [ ]:
from IPython.display import Image  # noqa: PLC0415
Image(filename=str(REPO / "results/figures/shap_summary.png"))

**Key insight.** The two Phase 2.1 gnomAD constraint features
(`lof_z` and `oe_mis_upper`) ranked #3 and #8 in global importance —
validating the hypothesis that gene-level priors carry orthogonal
signal to variant-level features.

Categorical features (the one-hot `alt_aa_P` is #4) also contribute:
Proline substitutions are known structural disruptors because
proline breaks α-helices.

## Per-variant explanations — worked example

Pick a highly-confident pathogenic prediction and show the SHAP
attributions that drove it.

In [ ]:
preds = pd.read_parquet(REPO / "results/metrics/xgboost_predictions.parquet")
preds_test = preds[preds["split"] == "test"]
shap_preds = shap_df.merge(preds_test[["variant_key", "p_calibrated", "y_true"]],
                            on="variant_key", how="inner")
confident_path = shap_preds[(shap_preds["y_true"] == 1) & (shap_preds["p_calibrated"] > 0.95)]
if len(confident_path) == 0:
    print("no confidently-correct pathogenic case in sample")
else:
    ex = confident_path.iloc[0]
    print(f"variant_key: {ex['variant_key']}")
    print(f"p_calibrated: {ex['p_calibrated']:.4f}  y_true: {int(ex['y_true'])}")
    row_shap = ex[feature_cols].abs().sort_values(ascending=False).head(5)
    print("\nTop-5 SHAP contributors (|value|, sign):")
    for fname in row_shap.index:
        val = ex[fname]
        direction = "↑ pathogenic" if val > 0 else "↓ benign"
        print(f"  {fname:<40s}  {val:+.4f}  {direction}")

## Error analysis — confident mistakes

326 variants (16.3 % of the sample) met the
`|p_calibrated − y_true| > 0.5` threshold — i.e. the model was
confidently wrong. The split is imbalanced: 264 false negatives vs
62 false positives. That FN-heavy pattern is typical for a
pathogenic-minority classifier and tells us exactly what to attack
next in Phase 2.

In [ ]:
errs = pd.read_csv(REPO / "results/metrics/confident_errors.csv")
print(f"Total confident errors: {len(errs):,}")
print(f"  false negatives (missed pathogenic): {(errs['error_type'] == 'false_negative').sum()}")
print(f"  false positives (benign called pathogenic): {(errs['error_type'] == 'false_positive').sum()}")
print()
print("Top features driving confident false-negatives:")
fn = errs[errs["error_type"] == "false_negative"]
top_fn = fn["top1_feature"].value_counts().head(6)
print(top_fn.to_string())

### Worked example of a confident false negative

In [ ]:
if len(fn):
    case = fn.iloc[0]
    print(f"variant_key: {case['variant_key']}")
    print(f"gene:        {case['gene']}")
    print(f"y_true=1, p_calibrated={case['p_calibrated']:.3f} → missed")
    print("Top-3 features that pushed the score *down* (toward benign):")
    for i in range(1, 4):
        f = case[f"top{i}_feature"]
        s = case[f"top{i}_shap"]
        print(f"  {f:<40s}  {s:+.4f}")

## Reproduce

```bash
python scripts/compute_shap.py
```

Output artifacts:
* `results/metrics/shap_values_test.parquet`
* `results/metrics/confident_errors.csv`
* `results/figures/shap_summary.png`
* `results/figures/shap_bar.png`
* `results/figures/shap_dependence_top3.png`